# Verification of mod 125 classification of eigenforms
Working precision is **625** throughout; the classification modulus is **125**.
The complete signed direct Manin sources, including torsion, are used.

For the faster option, leave `USE_ARCHIVED_SOURCE_DATA = True`. The Nim verifier reuses the stored Hecke matrices and transfer/complement maps. Build it once from the repository terminal with `./build_verify_hecke_relations.sh`.

Degrees are checked **in ascending order**, with the lower untwisted plus cases first, followed by all four orientations of the induction base. A single native session retains successful recursive lower checks. Archived Hecke constructions remain trusted producer inputs; transfer compatibility, spanning, and the relation equations are checked by the verifier. Missing low minus archives trigger a full check on the current source, not an unproved recursive shortcut.

The parameter table is in `p5_mod125_relation_data.json`. The source sandwiches are literal products of presented linear relations, not products of arbitrarily chosen divided source matrices. The notebook stops on a failed or inconclusive result; neither partial coverage nor a process error establishes classification.


In [1]:
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from sage.all import *
from verify_hecke_relations import (
    NimRelationVerifier,
    verify_hecke_relations_nim,
)
from p5_mod125 import (
    load_mod125_parameters,
    mod125_polynomials,
    mod125_relation_spec,
    mod125_possible_signatures,
    load_mod125_strong_representatives,
)

USE_ARCHIVED_SOURCE_DATA = True
SOURCE_DATA_DIRECTORY = Path("source_data")
MOD125_SOURCE_ARCHIVE = SOURCE_DATA_DIRECTORY / "p5_mod625_recursive"
STRONG_SIGNATURE_DIRECTORY = Path("strong_signatures/p5_m3")

# None permits a long native check. Interrupting this cell stops its process.
VERIFICATION_TIMEOUT = None
MAX_HOWELL_DIMENSION = 4096


In [2]:
p = 5
m = 4
modulus = p^m
R = Integers(modulus)

classification_exponent = 3
classification_modulus = p^classification_exponent
period = euler_phi(classification_modulus)

a_m = p^m * (p - 1)
b_m = p^(m - 1) * (p + 1)
surjectivity_bound = a_m + b_m

exact_induction_base = tuple(range(b_m, surjectivity_bound, 2))
lower_base_degrees = tuple(range(0, b_m, 2))
degree_residues = tuple(range(0, period, 2))

parameters = load_mod125_parameters()
polynomials = {r: mod125_polynomials(parameters[r]) for r in degree_residues}
native_relations = {
    r: mod125_relation_spec(parameters[r], MAX_HOWELL_DIMENSION)
    for r in degree_residues
}

# Both Dickson transitions preserve the parameter row r = d + 50q mod 100.
assert a_m % period == 0
assert (-b_m + 50) % period == 0
assert (50 * (p - 1)) % period == 0
assert power_mod(2, 125*(p - 1), modulus) == 1
assert power_mod(19, 125*(p - 1), modulus) == 1

print("Working modulus:", modulus)
print("Classification modulus:", classification_modulus)
print("Dickson degrees:", a_m, b_m)
print("degree residues:", degree_residues)
print("lower verification range:", lower_base_degrees)
print("exact induction base:", exact_induction_base)


Working modulus: 625
Classification modulus: 125
Dickson degrees: 2500 750
degree residues: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98)
lower verification range: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 

In [3]:
def verify_mod125_case(d, q, session):
    """Verify the displayed modulo-125 relation presentation in one degree and orientation.

    Use the configured source archive and native recursive verification; the
    returned report concerns finite-source equations, not a new all-weight proof.
    """
    relation_residue = (d + 50*q) % period

    compute = {
        "prime": p,
        "exponent": m,
        "degree": d,
        "orientation": q,
        "recursive": True,
        "recursive_verification": True,
    }
    if USE_ARCHIVED_SOURCE_DATA:
        compute["archive_directory"] = str(MOD125_SOURCE_ARCHIVE)
        compute["allow_missing_lower_orientations"] = True

    report = verify_hecke_relations_nim(
        native_relations[relation_residue],
        compute=compute,
        session=session,
        timeout=VERIFICATION_TIMEOUT,
    )
    checks = {check["name"]: check for check in report["relations"]}
    expected = {
        "ordinary_joint", "V_full_domain", "B_full_domain",
        "selector_0", "selector_1", "selector_2", "selector_3", "selector_4",
    }

    return {
        "degree": d,
        "orientation": q,
        "sign": (-1)^q,
        "relation_residue": relation_residue,
        "rank": report["rank"],
        "joint": checks.get("ordinary_joint"),
        "first_division": checks.get("V_full_domain"),
        "second_division": checks.get("B_full_domain"),
        "selectors": {ell: checks.get(f"selector_{ell}") for ell in range(5)},
        "native": report,
        "passed": (
            report["state"] == "passed"
            and set(checks) == expected
            and all(check["passed"] is True for check in checks.values())
        ),
    }


In [5]:
cases = (
    [(d, 0) for d in sorted(lower_base_degrees)]
    + [
        (d, q)
        for d in sorted(exact_induction_base)
        for q in range(0, p - 1)
    ]
)
assert all(cases[i][0] <= cases[i+1][0] for i in range(len(cases)-1))
assert len(cases) == 5375

results = []
with NimRelationVerifier() as session:
    for d, q in cases:
        print(f"checking d={d}, q={q} ...", flush=True)
        test = verify_mod125_case(d, q, session)
        results.append(test)

        def state(check):
            """Return the status field of this relation result, accepting the recorded report formats."""
            return check["state"] if check is not None else "not_run"

        print(
            f"d={d:4d}, "
            f"r={test['relation_residue']:2d}, "
            f"q={q}, "
            f"sign={test['sign']:+d}, "
            f"rank={test['rank']:3d}, "
            f"joint={state(test['joint'])}, "
            f"V={state(test['first_division'])}, "
            f"B={state(test['second_division'])}, "
            f"selectors={[state(test['selectors'][ell]) for ell in range(5)]}, "
            f"route={test['native']['recursive_route']}, "
            f"cached={test['native'].get('verification_cache_hit', False)}, "
            f"passed={test['passed']}",
            flush=True,
        )
        if not test["passed"]:
            print(test["native"])
            raise AssertionError("verification did not pass; inspect failed versus inconclusive")

completed = {(test["degree"], test["orientation"]) for test in results if test["passed"]}
low_degree_coverage = all((d, 0) in completed for d in lower_base_degrees)
induction_base_coverage = all(
    (d, q) in completed
    for d in exact_induction_base
    for q in range(0, p - 1)
)
assert low_degree_coverage and induction_base_coverage

print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("low-degree coverage:", low_degree_coverage)
print("induction-base coverage:", induction_base_coverage)
print("ALL PRESCRIBED MODULO-125 SOURCE IDENTITIES VERIFIED")


checking d=0, q=0 ...


d=   0, r= 0, q=0, sign=+1, rank=  0, joint=passed, V=passed, B=passed, selectors=['passed', 'passed', 'passed', 'passed', 'passed'], route=direct_base, cached=False, passed=True
checking d=2, q=0 ...
d=   2, r= 2, q=0, sign=+1, rank=  1, joint=passed, V=passed, B=passed, selectors=['passed', 'passed', 'passed', 'passed', 'passed'], route=direct_base, cached=False, passed=True
checking d=4, q=0 ...
d=   4, r= 4, q=0, sign=+1, rank=  1, joint=passed, V=passed, B=passed, selectors=['passed', 'passed', 'passed', 'passed', 'passed'], route=direct_base, cached=False, passed=True
checking d=6, q=0 ...
d=   6, r= 6, q=0, sign=+1, rank=  2, joint=passed, V=passed, B=passed, selectors=['passed', 'passed', 'passed', 'passed', 'passed'], route=direct_base, cached=False, passed=True
checking d=8, q=0 ...
d=   8, r= 8, q=0, sign=+1, rank=  1, joint=passed, V=passed, B=passed, selectors=['passed', 'passed', 'passed', 'passed', 'passed'], route=direct_base, cached=False, passed=True
checking d=10, q=

AssertionError: verification did not pass; inspect failed versus inconclusive

## From the source identities to the allowed signatures

The checks are the ordinary joint congruence modulo \(125\), full domain of
\[
\mathcal V=\mathfrak D_{t_{19},5},\qquad
\mathcal B_r=\mathfrak D_{1,25}N_r(\mathcal V),
\]
and the five terminal congruences
\[
P_{r,\ell}(\mathcal E_\ell\mathcal B_r\mathcal E_\ell)\mathcal E_\ell
\equiv0\pmod5.
\]
Here \(\mathcal E_\ell=E_\ell(\mathcal V)\), \(E_\ell(X)=(1-(X-\ell)^4)^5\);
all polynomials are expanded. \(P_{r,\ell}\) is lifted coefficientwise only
after taking \(R_{r,\ell}^4\) in \(\mathbf F_5[Y]\).
The displayed presentations, including both selector copies, are retained on the source.

Section 3 propagates these presented relations because both transitions preserve
\(r=d+50q\bmod100\). On the free relative target, successive cancellation gives
\(V=T_{19}/5\) and \(B_r=N_r(V)/25\), with retained precision
\(625\to125\to5\). This nested cancellation follows the proof method of the
division proposition; it is not a single application of its simultaneous-division
statement. Integral polynomial powers cause no further precision loss.
Since \((V^5-V)^2=0\bmod5\), the \(E_\ell(V)\) are orthogonal idempotents modulo
\(5\). Only on this target may the sandwiches be simplified. Saturation then
restricts the resulting operators to the cuspidal lattice.

For scalar eigenvalues \(v,b\), integrality of \(b\) in
\[
(v^5-v)^2-5A_r(v)(v^5-v)=25b
\]
forces \(u=(v^5-v)/5\) to be integral by valuation comparison.
Thus \(\bar v\in\mathbf F_5\) and \(\bar b=\bar u^2-A_r(\bar v)\bar u\).
Writing \(\lambda=v_0\bmod5\), the terminal conditions exclude
\(\bar v\notin\{\lambda,\lambda\pm2\}\), force \(\bar u=u_0\) when
\(\bar v=\lambda\), and force \(\bar u\in\mathbf F_5\) otherwise. The last
assertion works over **any residue-field extension**, using
\[
\prod_{t\in\mathbf F_5}(U^2-aU-t^2+at)=(U^5-U)^2.
\]
Since the difference quotient of \(X^5-X\) at two elements with the same
residue is a unit, these conditions give exactly the eleven allowed KRW
values \(t\) of \(v\) modulo \(25\). Hence \(a_{19}\equiv_{\rm val}5t\pmod{125}\).
The ordinary relation then gives
\(a_2^2\equiv_{\rm val}h_0+5h_1t+25h_2t^2\pmod{125}\).
Its two unit roots are distinct modulo \(5\), so \(a_2\) is KRW-congruent to
one of them, even with arbitrary ramification.

The source checks give the upper bound. Strong representatives are checked
separately below; they are not inferred from source characters.


# Check that each possible signature given by the identities above is realized by a strong eigenform


In [ ]:
possible_signatures = mod125_possible_signatures(parameters)

print(f"{'k mod ' + str(period):<10} | (a_2, a_19) mod {classification_modulus}")
print("-" * 70)
for r in sorted(possible_signatures):
    pairs = ", ".join(
        f"({a2}, {a19})"
        for a2, a19 in sorted(possible_signatures[r])
    )
    print(f"{str(r):<10} | {pairs}")

assert all(len(pairs) == 22 for pairs in possible_signatures.values())


In [ ]:
# Check the polynomial identity over F_5, not just its five rational values.
K.<U> = PolynomialRing(GF(5))
for a in GF(5):
    assert prod(U^2-a*U-t^2+a*t for t in GF(5)) == (U^5-U)^2
print("RESIDUE-FIELD POLYNOMIAL IDENTITY VERIFIED")


In [ ]:
l1, l2 = 2, 19

# Reuse the independent strong-signature scan instead of recomputing newforms.
# This checks payload seals and the weight/orbit/place bindings of all
# representatives. It does not repeat the producer's number-field computations.
signatures = load_mod125_strong_representatives(STRONG_SIGNATURE_DIRECTORY)
B = max(k for k, a2, a19 in signatures)

print("saved strong representatives:", len(signatures))
print("largest representative weight:", B)


In [ ]:
signatures_by_weight_residue = {}

for k, a2, a19 in signatures:
    r = k % period
    if r not in signatures_by_weight_residue:
        signatures_by_weight_residue[r] = set()
    signatures_by_weight_residue[r].add((ZZ(a2), ZZ(a19)))


In [ ]:
print(
    f"{'k mod ' + str(period):<10} | "
    f"(a_{l1}, a_{l2}) mod {classification_modulus}"
)
print("-" * 70)
for r in sorted(signatures_by_weight_residue):
    pairs = ", ".join(
        f"({a1}, {a2})"
        for a1, a2 in sorted(signatures_by_weight_residue[r])
    )
    print(f"{str(r):<10} | {pairs}")


In [ ]:
assert possible_signatures == signatures_by_weight_residue
print("EVERY POSSIBLE MODULO-125 SIGNATURE HAS A MATCHING SAVED STRONG REPRESENTATIVE")

# A bounded strong scan alone does not prove the forward classification.
assert low_degree_coverage and induction_base_coverage
print("COMPLETE SOURCE COVERAGE AND STRONG REPRESENTATIVE MATCHING VERIFIED")
